# 08 — 模型验证与可视化

本 notebook 提供交互式模型验证工具：
1. 加载训练好的模型
2. 3D 骨架对比 (GT vs Pred)
3. 多尺度骨架分析
4. 逐节点误差分布
5. 时间序列动画
6. 实时仿真对比
7. 密度场点云可视化

In [15]:
import os, sys, glob
import numpy as np

# GPU 设置（必须在 import torch 之前）
CUDA_DEVICE = 0
os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_DEVICE)

import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 3090


## 1. 配置

修改以下路径来选择要验证的模型和数据。

In [16]:
# ========== 配置区 ==========
# 修改这些路径来验证不同的模型
CHECKPOINT = '../train_log/train_ms_scnf/exp_20260428_1/phase2/model/best_model.pt'
DATA_DIR = '../data/seq_rr_3d'

# 也可以手动指定模型类型和阶段（通常不需要，会自动检测）
# MODEL_TYPE = 'ms_scnf'  # 'ms_scnf', 'cmstnf', 'mstnf'
# PHASE = 2               # 0, 1, 2

print(f'Checkpoint: {CHECKPOINT}')
print(f'Data dir:   {DATA_DIR}')
print(f'Exists:     ckpt={os.path.exists(CHECKPOINT)}, data={os.path.exists(DATA_DIR)}')

Checkpoint: ../train_log/train_ms_scnf/exp_20260428_1/phase2/model/best_model.pt
Data dir:   ../data/seq_rr_3d
Exists:     ckpt=True, data=True


## 2. 加载模型

使用 `model_loader` 自动检测模型类型并加载权重。

In [17]:
from src.utils.model_loader import load_model

info = load_model(CHECKPOINT, data_dir=DATA_DIR, device=device)
model = info['model']
print(f"\nModel type: {info['model_type']}")
print(f"Phase:      {info['phase']}")
print(f"Params:     {sum(p.numel() for p in model.parameters()):,}")

Loaded ms_scnf (phase 2) from ../train_log/train_ms_scnf/exp_20260428_1/phase2/model/best_model.pt
  action_dim=2, window_size=20, norm_factor=0.0050

Model type: ms_scnf
Phase:      2
Params:     269,261


In [18]:
from src.data.dataset import SoftSequenceDataset

ds = SoftSequenceDataset(DATA_DIR, seq_len=info['window_size'], return_3d=True)
print(f'Dataset: {len(ds)} samples, action_dim={ds.action_dim}, has_3d={ds.has_3d}')
print(f'Image size: {ds.H}x{ds.W}, focal={ds.focal}')
cam = ds.get_camera_params()
if cam:
    print(f'Camera: eye={cam["eye"]}, center={cam["center"]}')

Norm Factor: 0.005
Dataset: 5000 samples, action_dim=2, has_3d=True
Image size: 100x100, focal=136.4185228060769
Camera: eye=(1.5, 0.0, 0.5), center=(0.0, 0.0, 0.25)


## 3. 3D 骨架对比

随机采样多帧，对比 GT 与预测骨架。

In [19]:
from src.utils.skeleton_viz import plot_skeleton_3d, print_metrics

N_SAMPLES = 6
loader = DataLoader(ds, batch_size=N_SAMPLES, shuffle=True)
batch = next(iter(loader))

action_window = batch[0].to(device)
gt_pos = batch[-1]  # (B, 3, N)

with torch.no_grad():
    pred_dict = model.predict_skeleton(action_window)

pred_fine = pred_dict['fine'].cpu().numpy()

for i in range(N_SAMPLES):
    pred = pred_fine[i]
    gt = gt_pos[i].numpy().T
    metrics = print_metrics(pred, gt, label=f'Sample {i}')
    plot_skeleton_3d(pred, gt=gt, title=f'Sample {i}', show=True)

NameError: name 'torch' is not defined

## 4. 多尺度骨架分析

对比 coarse / medium / fine 三个尺度的预测。

In [ ]:
from src.utils.skeleton_viz import plot_multi_scale

# 取一个样本做多尺度分析
sample = ds[0]
aw = sample[0].unsqueeze(0).to(device)
gt = sample[-1].numpy().T  # (31, 3)

with torch.no_grad():
    pred_dict = model.predict_skeleton(aw)

# 转为 numpy
np_pred = {k: v[0].cpu().numpy() for k, v in pred_dict.items()}
print(f'Coarse:  {np_pred["coarse"].shape}')
print(f'Medium:  {np_pred["medium"].shape}')
print(f'Fine:    {np_pred["fine"].shape}')

plot_multi_scale(np_pred, gt=gt)

## 5. 逐节点误差分析

沿杆体方向显示每个节点的预测误差。

In [ ]:
from src.utils.skeleton_viz import plot_error_along_arm

# 统计多个样本的误差
N_STAT = 50
loader = DataLoader(ds, batch_size=8, shuffle=True)

all_errors = []
count = 0
with torch.no_grad():
    for batch in loader:
        if count >= N_STAT:
            break
        aw = batch[0].to(device)
        gt = batch[-1].numpy()  # (B, 3, N)
        pred = model.predict_skeleton(aw)['fine'].cpu().numpy()

        B = pred.shape[0]
        for b in range(B):
            if count >= N_STAT:
                break
            err = np.linalg.norm(pred[b] - gt[b].T, axis=1)
            all_errors.append(err)
            count += 1

all_errors = np.array(all_errors)  # (N_STAT, 31)
mean_errors = all_errors.mean(axis=0)
std_errors = all_errors.std(axis=0)

print(f'Statistics over {N_STAT} samples:')
print(f'  Mean MNE:   {mean_errors.mean():.6f} m')
print(f'  Max  MNE:   {mean_errors.max():.6f} m (at node {mean_errors.argmax()})')
print(f'  Tip  error: {mean_errors[-1]:.6f} m')
print(f'  Base error: {mean_errors[0]:.6f} m')

plot_error_along_arm(mean_errors, title=f'Mean Node-wise Error (n={N_STAT})')

## 6. 时间序列动画

按时间顺序预测连续帧，生成 GIF 动画。

In [ ]:
from src.utils.skeleton_viz import animate_skeleton_sequence

N_FRAMES = 80
n_frames = min(N_FRAMES, len(ds))

pred_seq = []
gt_seq = []
actions_seq = []

for i in range(n_frames):
    sample = ds[i]
    aw = sample[0].unsqueeze(0).to(device)
    gt = sample[-1].numpy().T
    raw_action = aw[0, -1].cpu().numpy() * ds.norm_factor

    with torch.no_grad():
        pred = model.predict_skeleton(aw)['fine'][0].cpu().numpy()

    pred_seq.append(pred)
    gt_seq.append(gt)
    actions_seq.append(raw_action)

pred_seq = np.stack(pred_seq)
gt_seq = np.stack(gt_seq)
actions_seq = np.stack(actions_seq)

save_path = 'output/notebook_animation.gif'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
animate_skeleton_sequence(pred_seq, gt_seq=gt_seq, actions=actions_seq,
                           save_path=save_path, fps=10)
print(f'Saved to {save_path}')

## 7. 实时仿真对比

创建仿真环境，逐步推进并对比模型预测与 GT。

In [ ]:
from elastica_env import ContinuousSoftArmEnv

env = ContinuousSoftArmEnv(dt=1e-4)
norm_factor = info['norm_factor']
window_size = info['window_size']
action_dim = info['action_dim']

history = np.zeros((1, window_size, action_dim))

N_LIVE_STEPS = 30
SIM_STEPS = 500  # 物理积分步/动作

live_pred = []
live_gt = []
live_actions = []
live_mne = []

print(f'Running {N_LIVE_STEPS} live simulation steps...')
for step in range(N_LIVE_STEPS):
    action = np.random.uniform(-0.3, 0.3, size=action_dim)
    env.set_action(action)
    for _ in range(SIM_STEPS):
        env.step(steps=1)

    _, _, positions, _ = env.get_observation_3d()
    gt = positions.T  # (31, 3)

    act_norm = action / norm_factor
    history = np.roll(history, -1, axis=1)
    history[0, -1] = act_norm
    aw_tensor = torch.from_numpy(history).float().to(device)

    with torch.no_grad():
        pred = model.predict_skeleton(aw_tensor)['fine'][0].cpu().numpy()

    mne = np.linalg.norm(pred - gt, axis=1).mean()
    live_pred.append(pred)
    live_gt.append(gt)
    live_actions.append(action)
    live_mne.append(mne)

    if step % 5 == 0:
        print(f'  Step {step:3d}/{N_LIVE_STEPS}: MNE={mne:.6f}m')

print(f'\nAverage MNE: {np.mean(live_mne):.6f}m')

In [ ]:
# 可视化实时仿真结果
live_pred = np.stack(live_pred)
live_gt = np.stack(live_gt)
live_actions = np.stack(live_actions)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# MNE 曲线
axes[0].plot(live_mne, 'r-o', markersize=3)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('MNE (m)')
axes[0].set_title('Mean Node Error over Time')
axes[0].axhline(np.mean(live_mne), color='k', linestyle='--', alpha=0.5)

# 驱动扭矩
for d in range(action_dim):
    axes[1].plot(live_actions[:, d], label=f'torque_{d}')
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Torque')
axes[1].legend()
axes[1].set_title('Driving Actions')

# 最终帧 GT vs Pred
ax3d = fig.add_subplot(133, projection='3d')
ax3d.plot(live_gt[-1][:, 0], live_gt[-1][:, 1], live_gt[-1][:, 2],
          'b-o', linewidth=3, markersize=4, label='GT')
ax3d.plot(live_pred[-1][:, 0], live_pred[-1][:, 1], live_pred[-1][:, 2],
          'r-o', linewidth=2, markersize=3, label='Pred')
ax3d.set_xlim(-0.3, 0.3); ax3d.set_ylim(-0.3, 0.3); ax3d.set_zlim(0, 0.6)
ax3d.set_title(f'Final Frame (MNE={live_mne[-1]:.4f}m)')
ax3d.legend()

plt.tight_layout()
plt.show()

## 8. 密度场点云可视化 (仅 MS-SCNF Phase 2)

从密度场中提取高密度点云，观察模型学习到的空间分布。

In [ ]:
if info['model_type'] == 'ms_scnf' and info['phase'] == 2:
    from src.utils.skeleton_viz import render_density_field

    # 取一个样本的动作
    sample = ds[0]
    aw = sample[0].unsqueeze(0).to(device)

    bounds = ((-0.3, 0.3), (-0.3, 0.3), (0.0, 0.6))
    cloud = render_density_field(model, aw, bounds, grid_res=25,
                                  threshold=0.3, device=device)

    print(f'Point cloud: {cloud.shape[0]} points')

    fig = plt.figure(figsize=(10, 5))
    ax1 = fig.add_subplot(121, projection='3d')
    ax1.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], c='red', s=1, alpha=0.5)
    ax1.set_xlim(-0.3, 0.3); ax1.set_ylim(-0.3, 0.3); ax1.set_zlim(0, 0.6)
    ax1.set_title('Density Field Point Cloud')

    # 与 GT 骨架叠加
    ax2 = fig.add_subplot(122, projection='3d')
    gt = sample[-1].numpy().T
    ax2.scatter(cloud[:, 0], cloud[:, 1], cloud[:, 2], c='red', s=1, alpha=0.3, label='Density')
    ax2.plot(gt[:, 0], gt[:, 1], gt[:, 2], 'b-o', linewidth=3, markersize=4, label='GT')
    ax2.set_xlim(-0.3, 0.3); ax2.set_ylim(-0.3, 0.3); ax2.set_zlim(0, 0.6)
    ax2.set_title('Density Field + GT Skeleton')
    ax2.legend()

    plt.tight_layout()
    plt.show()
else:
    print(f'Density field visualization requires MS-SCNF phase 2 model. Current: {info["model_type"]} phase {info["phase"]}')